# nightprints on Kaggle (free T4)

Settings before running: Accelerator = GPU T4, Internet = On.

Run top to bottom. Each step skips work it already did, so you can stop and resume. Everything is saved under /kaggle/working, which Kaggle keeps when you save a version.

In [ ]:
REPO_URL = "https://github.com/RutwikPatel13/nightprints.git"  # change if your repo name differs
STYLE = "configs/styles/dino.yaml"

In [ ]:
%cd /kaggle/working
!test -d nightprints || git clone $REPO_URL nightprints
%cd /kaggle/working/nightprints
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Generate candidates

60 prompts x 5 seeds = up to 300 images, about 2 to 3 hours. Use `--limit 6 --per-prompt 2` first for a quick test of the style prompt.

In [ ]:
!python scripts/generate_dataset.py --style $STYLE --per-prompt 5

## 2. Curate

Opens a public Gradio link. Keep 30 to 40 that share one consistent look. Stop this cell (the square button) when you're done.

In [ ]:
!python scripts/curate.py --style $STYLE --share

## 3. Caption and train

About 2 to 4 hours for 1500 steps. If it runs out of memory, change the second line to `!RES=768 bash scripts/train.sh`.

In [ ]:
!python scripts/caption.py --style $STYLE

In [ ]:
!STYLE=$STYLE bash scripts/train.sh

## 4. Eval

In [ ]:
!python scripts/eval.py --style $STYLE --lora outputs/lora/dino
from IPython.display import Markdown, Image, display
display(Markdown(open("outputs/eval/dino/report.md").read().split("![")[0]))
display(Image("outputs/eval/dino/grid_placement.png"))
display(Image("outputs/eval/dino/grid_pattern.png"))

## 5. Save the LoRA

Download this zip from the Output panel so you have the weights on your Mac.

In [ ]:
!cd outputs/lora && zip -qr /kaggle/working/dino_lora.zip dino/pytorch_lora_weights.safetensors && ls -lh /kaggle/working/dino_lora.zip

## 6. App for your friend

Prints a public link. Stop the cell to shut it down.

In [ ]:
!python app.py --style $STYLE --lora outputs/lora/dino --share